# RAG Variations: Exploring Different Retrieval-Augmented Generation Approaches


- **Dataset**: `neural-bridge/rag-dataset-1200` (HuggingFace)
- **Chunking**: Recursive Character Text Splitting
- **Embeddings**: `Qwen/Qwen3-Embedding-0.6B` via Sentence Transformers
- **Vector Store**: ChromaDB
- **LLM**: Gemma 3 4B via HuggingFace Transformers (4-bit quantized)

**RAG Strategies Covered:**
1. **Naive RAG** — Baseline retrieve-then-generate
2. **RAG + Re-ranking** — Cross-encoder re-ranks initial retrievals
3. **RAG + Query Decomposition** — Breaks complex queries into sub-questions
4. **RAG + HyDE** — Hypothetical Document Embeddings for better retrieval
5. **Agentic RAG** — Iterative retrieval loop with LLM-driven sufficiency checks
6. **RAG + Hybrid Search** — Fuses dense (embedding) and sparse (BM25) retrieval via RRF

## 1. Installations

Install all required Python packages.

> ⚠️ **Gemma 3 4B is a gated model on HuggingFace.** Before running you must:
> 1. Accept the license at [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it)
> 2. Authenticate: run `huggingface-cli login` in a terminal, or uncomment the login line in Section 6.

In [ ]:
# This may take some time to run, depending on your internet connection.
! pip install datasets chromadb sentence-transformers langchain langchain-community langchain-text-splitters rank-bm25
! pip install torch transformers accelerate bitsandbytes huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

## 2. Loading Libraries and Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import numpy as np
import re
from typing import List, Dict

In [ ]:
# preventing too many logs from the transformers library
import transformers
transformers.logging.set_verbosity_error()

### Dataset: `neural-bridge/rag-dataset-1200`

This dataset contains **1,200 question–context–answer triplets**. We use the `context` field to populate our vector store, and the `question` / `answer` fields to test and evaluate our RAG pipelines.

In [ ]:
print("Loading neural-bridge/rag-dataset-1200 ...")
dataset = load_dataset("neural-bridge/rag-dataset-1200", split="train")

print(f"Dataset loaded  : {len(dataset)} examples")
print(f"Features        : {list(dataset.features.keys())}")
print("\n \nSample entry:")
sample = dataset[0]
for key, val in sample.items():
    display_val = str(val)[:200] + "..." if len(str(val)) > 200 else str(val)
    print(f"  [{key}] {display_val}")

Loading neural-bridge/rag-dataset-1200 ...


README.md:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

data/train-00000-of-00001-f0c158413defd4(…): reconstructing file:   0%|          |  0.00B / 2.32MB            

data/train-00000-of-00001-f0c158413defd4(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-06d83c58a8ea10e(…): reconstructing file:   0%|          |  0.00B /  604kB            

data/test-00000-of-00001-06d83c58a8ea10e(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/960 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/240 [00:00<?, ? examples/s]

Dataset loaded  : 960 examples
Features        : ['context', 'question', 'answer']

 
Sample entry:
  [context] Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essa...
  [question] Who found the answer to a search query collar george herbert essay?
  [answer] Francisco Rogers found the answer to a search query collar george herbert essay.


## 3. Chunking

We apply **Recursive Character Text Splitting** to break document contexts into overlapping chunks.

| Parameter | Value |
|-----------|-------|
| Chunk size | 512 characters |
| Chunk overlap | 64 characters |
| Separators | `["\n\n", "\n", ". ", " ", ""]` |

The recursive strategy prioritises semantic boundaries (paragraphs → sentences → words), keeping related text together wherever possible.

In [ ]:
l = [1,2,3,4,5]
for index,num in enumerate(l):
    l[index] = l[index]+ 12
print(l)

l = [1,2,3,4,5]
l = [num+12 for num in l]
print(l)

[13, 14, 15, 16, 17]
[13, 14, 15, 16, 17]


In [ ]:
# Build Document objects from dataset contexts
# https://reference.langchain.com/python/langchain-community/document_loaders/hugging_face_dataset/HuggingFaceDatasetLoader$0
raw_docs = [
    Document(
        page_content=context,
        metadata={"source": f"doc_{index}", "doc_id": index}
    )
    for index, context in enumerate(dataset["context"])
]

# Recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(raw_docs)
chunk_lengths = [len(c.page_content) for c in chunks]

print(f"Original documents : {len(raw_docs)}")
print(f"Chunks produced    : {len(chunks)}")
print(f"Avg chunk size     : {np.mean(chunk_lengths):.0f} chars")
print(f"Min / Max          : {min(chunk_lengths)} / {max(chunk_lengths)} chars")

print("Example chunk:")
print("-" * 60)
print(chunks[0].page_content)

Original documents : 960
Chunks produced    : 9295
Avg chunk size     : 362 chars
Min / Max          : 1 / 512 chars
Example chunk:
------------------------------------------------------------
Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essay humility
business strategy case study solution
corporals course essay
decisions in paradise essays
college essay word count
credit cart terminal paper
byron don juan essay
democratic party essays
coursework language learning material teaching
christmas commercialized essay
dahrendorf essays theory society


In [ ]:
chunks[0]

Document(metadata={'source': 'doc_0', 'doc_id': 0}, page_content='Francisco Rogers found the answer to a search query collar george herbert essay\nLink ----> collar george herbert essay\nWrite my essay ESSAYERUDITE.COM\nconstitution research paper ideas\ndefinition essay humility\nbusiness strategy case study solution\ncorporals course essay\ndecisions in paradise essays\ncollege essay word count\ncredit cart terminal paper\nbyron don juan essay\ndemocratic party essays\ncoursework language learning material teaching\nchristmas commercialized essay\ndahrendorf essays theory society')

## 4. Setting up the Embedding Function

We use **`Qwen/Qwen3-Embedding-0.6B`** — a compact yet powerful embedding model from Alibaba's Qwen team.

- **Parameters**: 0.6 B
- **Task**: Retrieval-optimised text embedding
- **Special feature**: Supports an instruction prefix for queries (improves retrieval accuracy)

> ⚠️ The first run will download the model weights (~1.2 GB).

Homework: Switch the following code to LangChain

In [ ]:
# https://docs.langchain.com/oss/python/integrations/embeddings/sentence_transformers
print("Loading Qwen/Qwen3-Embedding-0.6B ...")
embed_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Embedding model loaded")

# Sanity check — use encode_query() which applies the model's built-in query prompt
test_emb = embed_model.encode_query(
    "What is retrieval augmented generation?",
    normalize_embeddings=True,
)
print(f"  Embedding dimension : {len(test_emb)}")


def encode_query(query: str) -> List[float]:
    """Embed a query using the model's built-in 'query' prompt (no manual prefix needed)."""
    return embed_model.encode_query(
        query,
        normalize_embeddings=True,
    ).tolist()


def encode_documents(texts: List[str], batch_size: int = 64) -> np.ndarray:
    """Embed document chunks using the model's built-in 'document' prompt."""
    return embed_model.encode_document(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

Loading Qwen/Qwen3-Embedding-0.6B ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model loaded
  Embedding dimension : 1024


## 5. Setting up the Vector Store

We use **ChromaDB** as our in-memory vector database.

- All chunks are embedded with `Qwen3-Embedding-0.6B` and stored in a ChromaDB collection
- Distance metric: **cosine** (works well with normalised embeddings)
- A `retrieve()` helper wraps the ChromaDB query for reuse across all RAG methods

*This will take approximately 30 minutes to run.*

Homework: Change to LangChain Code

In [ ]:
# EphemeralClient is the current in-memory API (chromadb.Client() is deprecated)
chroma_client = chromadb.EphemeralClient()

# configuration= is the modern way to set HNSW params (replaces metadata= dict)
collection = chroma_client.get_or_create_collection(
    name="rag_variations",
    configuration={"hnsw": {"space": "cosine"}},
)

# Prepare data
chunk_texts     = [c.page_content for c in chunks]
chunk_ids       = [f"chunk_{i}" for i in range(len(chunks))]
chunk_metadatas = [
    {
        "source": c.metadata.get("source", ""),
        "doc_id": str(c.metadata.get("doc_id", "")),
    }
    for c in chunks
]

# Embed all chunks
print("Computing embeddings for all chunks (this may take some time) ...")
chunk_embeddings = encode_documents(chunk_texts)
print(f"✓ Computed {len(chunk_embeddings)} embeddings")

# Insert into ChromaDB in batches of 500
BATCH = 500
print("\nInserting into ChromaDB ...")
for start in range(0, len(chunks), BATCH):
    end = min(start + BATCH, len(chunks))
    collection.add(
        documents  = chunk_texts[start:end],
        embeddings = chunk_embeddings[start:end].tolist(),
        ids        = chunk_ids[start:end],
        metadatas  = chunk_metadatas[start:end],
    )
    print(f"  {end}/{len(chunks)} chunks indexed")

print(f"\n✓ ChromaDB ready — {collection.count()} documents indexed")


def retrieve(query: str, n_results: int = 5) -> Dict:
    """Return the top-n most relevant chunks for a query."""
    results = collection.query(
        query_embeddings=[encode_query(query)],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )
    return results

Computing embeddings for all chunks (this may take some time) ...


Batches:   0%|          | 0/146 [00:00<?, ?it/s]

✓ Computed 9295 embeddings

Inserting into ChromaDB ...
  500/9295 chunks indexed
  1000/9295 chunks indexed
  1500/9295 chunks indexed
  2000/9295 chunks indexed
  2500/9295 chunks indexed
  3000/9295 chunks indexed
  3500/9295 chunks indexed
  4000/9295 chunks indexed
  4500/9295 chunks indexed
  5000/9295 chunks indexed
  5500/9295 chunks indexed
  6000/9295 chunks indexed
  6500/9295 chunks indexed
  7000/9295 chunks indexed
  7500/9295 chunks indexed
  8000/9295 chunks indexed
  8500/9295 chunks indexed
  9000/9295 chunks indexed
  9295/9295 chunks indexed

✓ ChromaDB ready — 9295 documents indexed


## 6. Implementing Naive RAG with Gemma3

**Naive (Simple) RAG** is the baseline approach:

```
Query → [Embed] → [ChromaDB top-k] → [Concatenate as context] → [Gemma 3 4B] → Answer
```

Steps:
1. **Retrieve** – embed the query; fetch top-k chunks from ChromaDB
2. **Augment** – inject retrieved chunks into the prompt as context
3. **Generate** – send the augmented prompt to Gemma 3 4B via HuggingFace Transformers

The model is loaded once here and shared by all RAG methods in the notebook.

> ⚠️ Gemma 3 4B requires ~8 GB VRAM in bfloat16. Enable 4-bit quantization (see comments) to reduce this to ~4 GB.

In [ ]:
from huggingface_hub import login; login()

In [ ]:
# fp 32bit -> fp 4 bit

In [ ]:
# ── Load Gemma 3 4B from HuggingFace ────────────────────────────────────────

MODEL_ID = "google/gemma-3-4b-it" # gemma 3 is a gated model

# Optional 4-bit quantization — reduces VRAM from ~8 GB to ~4 GB
from transformers import BitsAndBytesConfig
quant_cfg = BitsAndBytesConfig(load_in_4bit=True)

print(f"Loading {MODEL_ID} ...")
_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quant_cfg,
)

llm_pipeline = pipeline(
    "text-generation",
    model=_model,
    tokenizer=_tokenizer,
)
# device_map="auto" may distribute across multiple devices; hf_device_map shows the layout
device_info = getattr(_model, "hf_device_map", str(next(_model.parameters()).device))
print(f"✓ {MODEL_ID} loaded  |  device map: {device_info}")


def generate(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate a response using Gemma 3 4B via HuggingFace Transformers."""
    outputs = llm_pipeline(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return outputs[0]["generated_text"][-1]["content"]


def naive_rag(query: str, n_results: int = 5) -> Dict:
    """Baseline RAG: retrieve top-k chunks, build context, generate answer."""
    results = retrieve(query, n_results)
    docs    = results["documents"][0]
    dists   = results["distances"][0]

    context = "\n\n---\n\n".join(docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":          query,
        "answer":         generate(prompt),
        "retrieved_docs": docs,
        "distances":      dists,
    }


# ── Quick test ──────────────────────────────────────────────────────────────
test_query   = dataset[0]["question"]
ground_truth = dataset[0]["answer"]

print(f"Query        : {test_query}")
print(f"Ground truth : {ground_truth}")
print("=" * 70)

result_naive = naive_rag(test_query)
print(f"\n[Naive RAG] Answer:\n{result_naive['answer']}")
print(f"\nTop retrieved chunk (distance={result_naive['distances'][0]:.4f}):")
print(result_naive["retrieved_docs"][0][:300] + "...")

Loading google/gemma-3-4b-it ...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✓ google/gemma-3-4b-it loaded  |  device map: cuda:0
Query        : Who found the answer to a search query collar george herbert essay?
Ground truth : Francisco Rogers found the answer to a search query collar george herbert essay.

[Naive RAG] Answer:
Francisco Rogers found the answer to a search query "collar george herbert essay".

Top retrieved chunk (distance=0.2331):
Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essay humility
business strategy case study solution
corporals course essay
decisions in paradise essays...


## 7a. Understanding RAG with Re-ranking

### The Problem

Naive retrieval uses a **bi-encoder**: query and document are embedded independently; relevance is measured by cosine similarity. This is fast and scalable, but misses fine-grained relevance signals that require *joint* reasoning over both the query and the document.

### Solution: Cross-Encoder Re-ranking

Re-ranking is a **two-stage** pipeline:

| Stage | Model type | Goal |
|-------|-----------|------|
| 1. Candidate retrieval | Bi-encoder (Qwen3-Embedding) | High recall — fetch many candidates quickly |
| 2. Re-ranking | Cross-encoder | High precision — score each (query, doc) pair jointly |

A **cross-encoder** reads query and document *together*, enabling much richer relevance judgements at the cost of higher latency.

### Pipeline

```
Query
  │
  ├─[Bi-encoder]──▶ Top-20 candidates
  │                        │
  │───────────────▶ [Cross-encoder score each pair] (fast)
  │                        │
  └───────────────▶ Top-5 re-ranked docs ──▶ [Gemma 3 4B] ──▶ Answer
```

We use **`cross-encoder/ms-marco-MiniLM-L-6-v2`** — trained on the MS MARCO passage-ranking dataset.

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
scores = model.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])
print(scores)
# [ 8.607138 -4.320078]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[ 8.607139  -4.3200774]


In [ ]:
print("Loading cross-encoder/ms-marco-MiniLM-L-6-v2 ...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✓ Cross-encoder loaded")


def rag_with_reranking(query: str, initial_k: int = 20, top_n: int = 5) -> Dict:
    """
    Two-stage retrieval:
      Stage 1 – bi-encoder retrieves `initial_k` candidates
      Stage 2 – cross-encoder re-ranks and keeps the best `top_n`
    """
    # Stage 1: broad bi-encoder retrieval
    results    = retrieve(query, initial_k)
    candidates = results["documents"][0]

    # Stage 2: cross-encoder scoring
    pairs         = [(query, doc) for doc in candidates]
    rerank_scores = cross_encoder.predict(pairs)

    # Keep top_n by cross-encoder score (higher = more relevant)
    ranked_idx    = np.argsort(rerank_scores)[::-1][:top_n]
    reranked_docs = [candidates[i] for i in ranked_idx]
    reranked_scrs = [float(rerank_scores[i]) for i in ranked_idx]

    context = "\n\n---\n\n".join(reranked_docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":            query,
        "answer":           generate(prompt),
        "reranked_docs":    reranked_docs,
        "reranking_scores": reranked_scrs,
        "initial_k":        initial_k,
    }

Loading cross-encoder/ms-marco-MiniLM-L-6-v2 ...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Cross-encoder loaded


## 7b. Implementing RAG with Re-ranking

We run the two-stage pipeline on our test query and compare the retrieved documents and answer with those from Naive RAG.

In [ ]:
result_rerank = rag_with_reranking(test_query, initial_k=20, top_n=5)

print(f"Query: {test_query}")
print("=" * 70)
print(f"\n[RAG + Re-ranking] Answer:\n{result_rerank['answer']}")
print(f"\nInitial candidates : {result_rerank['initial_k']}")
print(f"After re-ranking   : {len(result_rerank['reranked_docs'])} docs kept")
print(f"\nTop re-ranked doc (score={result_rerank['reranking_scores'][0]:.4f}):")
print(result_rerank["reranked_docs"][0][:300] + "...")
print()
print("─" * 70)
print("Answer comparison on test query:")
print(f"  Naive RAG   : {result_naive['answer'][:200]}")
print(f"  + Re-ranking: {result_rerank['answer'][:200]}")

Query: Who found the answer to a search query collar george herbert essay?

[RAG + Re-ranking] Answer:
Francisco Rogers found the answer to a search query collar george herbert essay.

Initial candidates : 20
After re-ranking   : 5 docs kept

Top re-ranked doc (score=9.1580):
Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essay humility
business strategy case study solution
corporals course essay
decisions in paradise essays...

──────────────────────────────────────────────────────────────────────
Answer comparison on test query:
  Naive RAG   : Francisco Rogers found the answer to a search query "collar george herbert essay".
  + Re-ranking: Francisco Rogers found the answer to a search query collar george herbert essay.


## 7c. Evaluating Improvement in Results due to Re-ranking

We evaluate both approaches on **5 test samples** using **Word-Overlap F1** — a simple reference-based metric that computes the token-level F1 score between the generated answer and the ground-truth answer.

> A higher score indicates greater lexical overlap with the ground truth. This metric is a proxy; real-world evaluation should also include semantic similarity and human judgement.

In [ ]:
def word_overlap_f1(prediction: str, reference: str) -> float:
    """Token-level F1 score between prediction and reference (case-insensitive)."""
    pred_tokens = set(re.findall(r'\w+', prediction.lower()))
    ref_tokens  = set(re.findall(r'\w+', reference.lower()))
    if not pred_tokens or not ref_tokens:
        return 0.0
    common    = pred_tokens & ref_tokens
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# Evaluation samples
EVAL_SAMPLES = [dataset[i] for i in range(5)]

# Shared results dict – populated across evaluation sections
eval_results: Dict[str, List[float]] = {}

print("Evaluating Naive RAG vs RAG + Re-ranking (5 samples) ...\n")
print("=" * 70)

naive_f1s  = []
rerank_f1s = []

for i, sample in enumerate(EVAL_SAMPLES):
    q        = sample["question"]
    expected = sample["answer"]

    r_naive  = naive_rag(q, n_results=5)
    r_rerank = rag_with_reranking(q, initial_k=20, top_n=5)

    f1_naive  = word_overlap_f1(r_naive["answer"],  expected)
    f1_rerank = word_overlap_f1(r_rerank["answer"], expected)

    naive_f1s.append(f1_naive)
    rerank_f1s.append(f1_rerank)

    print(f"Sample {i+1}: {q[:70]}...")
    print(f"  Ground truth : {expected[:100]}")
    print(f"  Naive RAG    : {r_naive['answer'][:100]}  [F1={f1_naive:.3f}]")
    print(f"  + Re-ranking : {r_rerank['answer'][:100]}  [F1={f1_rerank:.3f}]")
    print()

eval_results["Naive RAG"]  = naive_f1s
eval_results["Re-ranking"] = rerank_f1s

print("=" * 70)
print(f"Avg Word-Overlap F1:")
print(f"  Naive RAG    : {np.mean(naive_f1s):.3f}")
print(f"  + Re-ranking : {np.mean(rerank_f1s):.3f}  (Δ {np.mean(rerank_f1s) - np.mean(naive_f1s):+.3f})")

Evaluating Naive RAG vs RAG + Re-ranking (5 samples) ...

Sample 1: Who found the answer to a search query collar george herbert essay?...
  Ground truth : Francisco Rogers found the answer to a search query collar george herbert essay.
  Naive RAG    : Francisco Rogers found the answer to a search query "collar george herbert essay".  [F1=1.000]
  + Re-ranking : Francisco Rogers found the answer to a search query collar george herbert essay.  [F1=1.000]

Sample 2: What are some of the potential negative impacts of charity as discusse...
  Ground truth : The context discusses that charity can sometimes exacerbate problems rather than solve them. It can 
  Naive RAG    : According to the text, some of the potential negative impacts of charity are:

*   It can be a "band  [F1=0.286]
  + Re-ranking : According to the text, some of the potential negative impacts of charity are:

*   It's a "bandaid p  [F1=0.265]

Sample 3: Who were the three stars in the NHL game between Buffalo Sabres and

## 8a. Understanding RAG with Query Decomposition

### The Problem

A single query embedding may not capture all facets of a **multi-aspect question**.

**Example**: *"How do transformers and recurrent networks compare in language modelling, and what are their key trade-offs?"*

This question spans: (1) transformer architecture, (2) recurrent networks, (3) language modelling, and (4) trade-off analysis. A single embedding may only retrieve documents relevant to one or two of those aspects.

### Solution: Query Decomposition

We ask the LLM to split the original query into **2–3 focused sub-questions**, retrieve documents for each, deduplicate, and then generate with the merged context.

### Pipeline

```
Complex Query
      │
  [Gemma 3 4B]──▶ Sub-question 1 ──▶ [Retrieval] ──▶ Docs₁
      │           Sub-question 2 ──▶ [Retrieval] ──▶ Docs₂
      │           Sub-question 3 ──▶ [Retrieval] ──▶ Docs₃
      │                                    │
      │                          [Deduplicate & Merge]
      │                                    │
      └───────────────────────────▶ [Gemma 3 4B] ──▶ Answer
```

**Benefits**: Better recall for multi-faceted queries; richer context  
**Trade-offs**: Extra LLM call for decomposition; higher latency

## 8b. Implementing RAG with Query Decomposition

We implement the decomposition pipeline and run it on our test query to observe how the sub-questions and retrieved evidence differ from naive retrieval.

In [ ]:
def decompose_query(query: str) -> List[str]:
    """Ask Gemma to break a complex query into 2–3 focused sub-questions."""
    prompt = (
        "Break the following question into 2-3 simpler, specific sub-questions. "
        "Each sub-question should target a single aspect that helps answer the original question. "
        "Return ONLY the sub-questions, one per line, with no numbering or extra text.\n\n"
        f"Original question: {query}"
    )
    response = generate(prompt)
    sub_qs   = [
        line.strip()
        for line in response.strip().split('\n')
        if line.strip() and len(line.strip()) > 10
    ]
    return sub_qs[:3]


def rag_with_query_decomposition(query: str, n_results: int = 4) -> Dict:
    """
    RAG with Query Decomposition:
      1. Decompose the query into sub-questions using the LLM
      2. Retrieve documents for each sub-question
      3. Deduplicate and merge retrieved docs (capped at 10)
      4. Generate answer from the merged context
    """
    # Step 1: decompose
    sub_queries = decompose_query(query)

    # Step 2 & 3: retrieve + deduplicate (preserving order)
    seen: Dict[str, bool] = {}
    for sq in sub_queries:
        results = retrieve(sq, n_results)
        for doc in results["documents"][0]:
            seen.setdefault(doc, True)

    unique_docs = list(seen.keys())[:10]

    # Step 4: generate
    context = "\n\n---\n\n".join(unique_docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":          query,
        "answer":         generate(prompt),
        "sub_queries":    sub_queries,
        "retrieved_docs": unique_docs,
    }


# ── Test ────────────────────────────────────────────────────────────────────
result_qd = rag_with_query_decomposition(test_query)

print(f"Query: {test_query}")
print("=" * 70)
print(f"\nDecomposed into {len(result_qd['sub_queries'])} sub-questions:")
for sq in result_qd["sub_queries"]:
    print(f"  • {sq}")

print(f"\nUnique docs retrieved : {len(result_qd['retrieved_docs'])}")
print(f"\n[RAG + Query Decomposition] Answer:\n{result_qd['answer']}")

Query: Who found the answer to a search query collar george herbert essay?

Decomposed into 3 sub-questions:
  • What was the search query?
  • Who is George Herbert?
  • What was the source of the essay?

Unique docs retrieved : 10

[RAG + Query Decomposition] Answer:
Francisco Rogers


## 8c. Evaluating Improvement in Results due to Query Decomposition

Same 5 test samples; same Word-Overlap F1 metric. We compare against the Naive RAG baseline scores stored in `eval_results["Naive RAG"]` from Section 7c.

In [ ]:
print("Evaluating Naive RAG vs RAG + Query Decomposition (5 samples) ...\n")
print("=" * 70)

qd_f1s = []

for i, sample in enumerate(EVAL_SAMPLES):
    q        = sample["question"]
    expected = sample["answer"]

    r_qd    = rag_with_query_decomposition(q, n_results=4)
    f1_qd   = word_overlap_f1(r_qd["answer"], expected)
    f1_naive = eval_results["Naive RAG"][i]   # reuse baseline from Section 7c

    qd_f1s.append(f1_qd)

    print(f"Sample {i+1}: {q[:70]}...")
    print(f"  Sub-questions  : {r_qd['sub_queries']}")
    print(f"  Ground truth   : {expected[:100]}")
    print(f"  Naive RAG      : [F1={f1_naive:.3f}]")
    print(f"  + Query Decomp : {r_qd['answer'][:100]}  [F1={f1_qd:.3f}]")
    print()

eval_results["Query Decomposition"] = qd_f1s

print("=" * 70)
print(f"Avg Word-Overlap F1:")
print(f"  Naive RAG             : {np.mean(eval_results['Naive RAG']):.3f}")
print(f"  + Query Decomposition : {np.mean(qd_f1s):.3f}  (Δ {np.mean(qd_f1s) - np.mean(eval_results['Naive RAG']):+.3f})")

Evaluating Naive RAG vs RAG + Query Decomposition (5 samples) ...

Sample 1: Who found the answer to a search query collar george herbert essay?...
  Sub-questions  : ['What was the search query?', 'Who is George Herbert?', 'What was the source of the essay?']
  Ground truth   : Francisco Rogers found the answer to a search query collar george herbert essay.
  Naive RAG      : [F1=1.000]
  + Query Decomp : Francisco Rogers  [F1=0.267]

Sample 2: What are some of the potential negative impacts of charity as discusse...
  Sub-questions  : ['What specific types of negative impacts are identified in the context?', 'Does the context discuss any power imbalances created by charitable organizations?', 'Does the context mention potential negative impacts on beneficiaries themselves?']
  Ground truth   : The context discusses that charity can sometimes exacerbate problems rather than solve them. It can 
  Naive RAG      : [F1=0.286]
  + Query Decomp : According to the context, some of the poten

## 9a. Understanding RAG with HyDE (Hypothetical Document Embeddings)

### The Vocabulary Mismatch Problem

Even with strong embedding models, there is often a **vocabulary gap** between:
- **Queries** — short, interrogative, keyword-like
- **Documents** — longer, declarative, domain-specific vocabulary

This mismatch can cause the query embedding to land in a different part of the embedding space from the relevant documents, degrading retrieval quality.

### Solution: HyDE (Gao et al., 2022)

**HyDE** addresses this by generating a *hypothetical* document that answers the query, then using **that document's embedding** as the retrieval key rather than the raw query embedding.

Key insight: a generated document is *lexically* and *stylistically* similar to real documents, so its embedding is a better proxy for what you want to retrieve.

### Pipeline

```
Query
  │
  [Gemma 3 4B] ──▶ Hypothetical Answer (dense, declarative text)
                              │
                    [Qwen3-Embedding encode]
                              │
                    [ChromaDB retrieval] ──▶ Real top-k docs
                                                    │
                                          [Gemma 3 4B] ──▶ Final Answer
```

**Benefits**: Bridges the query–document vocabulary gap; improves retrieval for knowledge-intensive questions  
**Trade-offs**: Extra LLM call; hypothetical doc quality affects retrieval; risk of hallucination-driven retrieval

## 9b. Implementing RAG with HyDE

We generate a hypothetical passage with Gemma, embed it with Qwen3-Embedding, retrieve real documents nearest to that embedding, and then generate the final answer from the *real* retrieved documents.

In [ ]:
def generate_hypothetical_doc(query: str) -> str:
    """Generate a short factual passage that would directly answer the query."""
    prompt = (
        "Write a short factual passage (3–4 sentences) that directly answers the question below. "
        "Write in the style of an encyclopedia entry — precise and informative. "
        "Do not include any preamble.\n\n"
        f"Question: {query}\n\nPassage:"
    )
    return generate(prompt)


def hyde_rag(query: str, n_results: int = 5) -> Dict:
    """
    HyDE RAG – Hypothetical Document Embeddings:
      1. Generate a hypothetical document with the LLM
      2. Embed the hypothetical doc as a DOCUMENT (no query prefix) — it sits in
         the same embedding space as real corpus documents, not the query space
      3. Retrieve real documents nearest to the hypothetical embedding
      4. Generate the final answer from the real retrieved documents
    """
    # Step 1: generate hypothetical document
    hypo_doc = generate_hypothetical_doc(query)

    # Step 2: embed as DOCUMENT — encode_document() uses no query instruction prefix
    hypo_emb = embed_model.encode_document(
        hypo_doc,
        normalize_embeddings=True,
    ).tolist()

    # Step 3: retrieve real documents closest to the hypothetical embedding
    results        = collection.query(
        query_embeddings=[hypo_emb],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )
    retrieved_docs = results["documents"][0]
    distances      = results["distances"][0]

    # Step 4: generate final answer from real docs
    context = "\n\n---\n\n".join(retrieved_docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":            query,
        "answer":           generate(prompt),
        "hypothetical_doc": hypo_doc,
        "retrieved_docs":   retrieved_docs,
        "distances":        distances,
    }


# ── Test ────────────────────────────────────────────────────────────────────
result_hyde = hyde_rag(test_query)

print(f"Query: {test_query}")
print("=" * 70)
print(f"\nHypothetical document generated:")
print(f"  {result_hyde['hypothetical_doc'][:400]}")
print(f"\nTop retrieved doc (distance={result_hyde['distances'][0]:.4f}):")
print(f"  {result_hyde['retrieved_docs'][0][:300]}...")
print(f"\n[HyDE RAG] Answer:\n{result_hyde['answer']}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Who found the answer to a search query collar george herbert essay?

Hypothetical document generated:
  George Herbert’s “Collar” was first identified as a key example of metaphysical poetry by John Nichols in 1846, following a thorough textual examination of the poem. Nichols’ analysis, published in *The Monthly Repository*, established the essay’s central themes of death and divine grace within the context of Herbert’s religious poetry. Subsequent scholars, including T.S. Eliot, have largely accep

Top retrieved doc (distance=0.5161):
  Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essay humility
business strategy case study solution
corporals course essay
decisions in paradise essays...

[HyDE RAG] Answer:
Francisco Rogers found the answer to a search query "collar george herbert essay".


## 9c. Evaluating Improvement in Results due to HyDE

Final evaluation of HyDE vs Naive RAG, followed by a **complete summary** comparing all four RAG strategies across the same 5 test samples.

In [ ]:
print("Evaluating Naive RAG vs HyDE (5 samples) ...\n")
print("=" * 70)

hyde_f1s = []

for i, sample in enumerate(EVAL_SAMPLES):
    q        = sample["question"]
    expected = sample["answer"]

    r_hyde   = hyde_rag(q, n_results=5)
    f1_hyde  = word_overlap_f1(r_hyde["answer"], expected)
    f1_naive = eval_results["Naive RAG"][i]

    hyde_f1s.append(f1_hyde)

    print(f"Sample {i+1}: {q[:70]}...")
    print(f"  Ground truth : {expected[:100]}")
    print(f"  Naive RAG    : [F1={f1_naive:.3f}]")
    print(f"  HyDE RAG     : {r_hyde['answer'][:100]}  [F1={f1_hyde:.3f}]")
    print()

eval_results["HyDE"] = hyde_f1s

print("=" * 70)
print(f"Avg Word-Overlap F1:")
print(f"  Naive RAG : {np.mean(eval_results['Naive RAG']):.3f}")
print(f"  + HyDE    : {np.mean(hyde_f1s):.3f}  (Δ {np.mean(hyde_f1s) - np.mean(eval_results['Naive RAG']):+.3f})")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating Naive RAG vs HyDE (5 samples) ...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 1: Who found the answer to a search query collar george herbert essay?...
  Ground truth : Francisco Rogers found the answer to a search query collar george herbert essay.
  Naive RAG    : [F1=1.000]
  HyDE RAG     : Francisco Rogers found the answer to a search query "collar george herbert essay".  [F1=1.000]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Sample 3: Who were the three stars in the NHL game between Buffalo Sabres and Ed...
  Ground truth : The three stars were Ryan O’Reilly, Brian Gionta, and Leon Draisaitl.
  Naive RAG    : [F1=0.737]
  HyDE RAG     : Ryan O’Reilly; Brian Gionta; Leon Draisaitl  [F1=0.737]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 4: What services does Pearl Moving Company in Santa Clarita, 91390 offer?...
  Ground truth : Pearl Moving Company Santa Clarita, 91390 offers services such as apartment moving, local moving, of
  Naive RAG    : [F1=0.459]
  HyDE RAG     : The context doesn't explicitly list *all* the services Pearl Moving Company in Santa Clarita, 91390   [F1=0.515]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 5: What are the responsibilities of a Senior Planning Engineer in London,...
  Ground truth : The responsibilities of a Senior Planning Engineer in London, United Kingdom include providing Integ
  Naive RAG    : [F1=0.593]
  HyDE RAG     : According to the provided text, a Senior Planning Engineer in London, United Kingdom is responsible   [F1=0.750]

Avg Word-Overlap F1:
  Naive RAG : 0.615
  + HyDE    : 0.647  (Δ +0.032)


## 10a. Understanding Agentic RAG

### The Problem with One-Shot Retrieval

All previous RAG variants retrieve **once** and generate. This fails when:
- A query is ambiguous and the first retrieval misses the point
- The answer requires evidence from multiple retrieval angles
- The initial results are topically close but factually insufficient

### Solution: Agentic RAG

**Agentic RAG** adds a reasoning loop: after each retrieval, the LLM evaluates whether the context is sufficient. If not, it reformulates the query and retrieves again — up to a configurable limit.

### Pipeline

```
Query
  │
  [Retrieve]
  │
  [Agent: context sufficient?] ──YES──▶ [Generate Answer]
  │ NO
  [Agent: suggest better query]
  │
  [Retrieve with refined query]
  │
  [Agent: context sufficient?] ──YES──▶ [Generate Answer]
  │ NO  (repeats up to max_iterations)
  ...
```

**Benefits**: Self-correcting; handles ambiguous and multi-hop queries; accumulates broader evidence  
**Trade-offs**: Multiple LLM calls per query; higher latency; query drift risk over many iterations

## 10b. Implementing Agentic RAG

Each iteration: retrieve → ask Gemma whether the accumulated context is sufficient → if not, ask Gemma to suggest a refined follow-up query → repeat. The final answer is generated from all evidence gathered across iterations.

In [ ]:
def agentic_rag(query: str, n_results: int = 5, max_iterations: int = 3) -> Dict:
    """
    Agentic RAG with iterative retrieval and LLM-driven sufficiency checks.
      1. Retrieve with current query
      2. LLM assesses whether accumulated context is sufficient
      3. If not, LLM suggests a refined follow-up query
      4. Repeat up to max_iterations
      5. Generate final answer from all accumulated context
    """
    all_docs: Dict[str, bool] = {}   # content → True (preserves insertion order, deduplicates)
    iterations_log = []
    current_query  = query

    for iteration in range(max_iterations):
        results  = retrieve(current_query, n_results)
        new_docs = results["documents"][0]
        for doc in new_docs:
            all_docs.setdefault(doc, True)

        context_excerpt = "\n\n---\n\n".join(list(all_docs.keys())[:5])

        check_prompt = (
            "You are a research assistant. Given the question and retrieved context below, "
            "decide if there is enough information to give a complete, accurate answer.\n\n"
            f"Question: {query}\n\n"
            f"Context:\n{context_excerpt[:800]}\n\n"
            "Reply with EXACTLY two lines and nothing else:\n"
            "SUFFICIENT: YES or NO\n"
            "FOLLOW_UP: a more specific search query if NO, else NONE"
        )
        assessment = generate(check_prompt, max_new_tokens=80)

        sufficient = "SUFFICIENT: YES" in assessment.upper()
        follow_up  = None
        for line in assessment.strip().split('\n'):
            if "FOLLOW_UP:" in line.upper():
                candidate = line.split(':', 1)[-1].strip()
                if candidate.upper() not in ("NONE", ""):
                    follow_up = candidate
                break

        iterations_log.append({
            "iteration":      iteration + 1,
            "query_used":     current_query,
            "docs_retrieved": len(new_docs),
            "sufficient":     sufficient,
        })

        if sufficient or follow_up is None:
            break
        current_query = follow_up

    # Generate final answer from all accumulated context
    final_context = "\n\n---\n\n".join(list(all_docs.keys())[:10])
    prompt = f"""Answer the question using only the context provided below.

Context:
{final_context}

Question: {query}

Answer:"""

    return {
        "query":          query,
        "answer":         generate(prompt),
        "iterations":     iterations_log,
        "retrieved_docs": list(all_docs.keys()),
    }


# ── Test ────────────────────────────────────────────────────────────────────
result_agentic = agentic_rag(test_query, n_results=5, max_iterations=3)

print(f"Query: {test_query}")
print("=" * 70)
print(f"\nAgent ran {len(result_agentic['iterations'])} iteration(s):")
for it in result_agentic["iterations"]:
    status = "✓ sufficient" if it["sufficient"] else "↺ needs more"
    print(f"  Iter {it['iteration']}: '{it['query_used'][:65]}...'  [{status}]")

print(f"\nTotal unique docs accumulated : {len(result_agentic['retrieved_docs'])}")
print(f"\n[Agentic RAG] Answer:\n{result_agentic['answer']}")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Who found the answer to a search query collar george herbert essay?

Agent ran 1 iteration(s):
  Iter 1: 'Who found the answer to a search query collar george herbert essa...'  [✓ sufficient]

Total unique docs accumulated : 5

[Agentic RAG] Answer:
Francisco Rogers found the answer to a search query "collar george herbert essay".


## 10c. Evaluating Improvement in Results due to Agentic RAG

Same 5 test samples; same Word-Overlap F1 metric. We also report the number of retrieval iterations used per sample.

In [ ]:
print("Evaluating Naive RAG vs Agentic RAG (5 samples) ...\n")
print("=" * 70)

agentic_f1s = []

for i, sample in enumerate(EVAL_SAMPLES):
    q        = sample["question"]
    expected = sample["answer"]

    r_agentic  = agentic_rag(q, n_results=5, max_iterations=3)
    f1_agentic = word_overlap_f1(r_agentic["answer"], expected)
    f1_naive   = eval_results["Naive RAG"][i]

    agentic_f1s.append(f1_agentic)

    print(f"Sample {i+1}: {q[:70]}...")
    print(f"  Ground truth  : {expected[:100]}")
    print(f"  Naive RAG     : [F1={f1_naive:.3f}]")
    iters = len(r_agentic["iterations"])
    print(f"  Agentic RAG   : {r_agentic['answer'][:100]}  [F1={f1_agentic:.3f}]  ({iters} iter(s))")
    print()

eval_results["Agentic RAG"] = agentic_f1s

print("=" * 70)
print(f"Avg Word-Overlap F1:")
print(f"  Naive RAG   : {np.mean(eval_results['Naive RAG']):.3f}")
print(f"  Agentic RAG : {np.mean(agentic_f1s):.3f}  (Δ {np.mean(agentic_f1s) - np.mean(eval_results['Naive RAG']):+.3f})")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating Naive RAG vs Agentic RAG (5 samples) ...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 1: Who found the answer to a search query collar george herbert essay?...
  Ground truth  : Francisco Rogers found the answer to a search query collar george herbert essay.
  Naive RAG     : [F1=1.000]
  Agentic RAG   : Francisco Rogers found the answer to a search query "collar george herbert essay".  [F1=1.000]  (1 iter(s))



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Sample 3: Who were the three stars in the NHL game between Buffalo Sabres and Ed...
  Ground truth  : The three stars were Ryan O’Reilly, Brian Gionta, and Leon Draisaitl.
  Naive RAG     : [F1=0.737]
  Agentic RAG   : Ryan O’Reilly; Brian Gionta; Leon Draisaitl  [F1=0.737]  (1 iter(s))



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 4: What services does Pearl Moving Company in Santa Clarita, 91390 offer?...
  Ground truth  : Pearl Moving Company Santa Clarita, 91390 offers services such as apartment moving, local moving, of
  Naive RAG     : [F1=0.459]
  Agentic RAG   : The context doesn't explicitly list *all* the services Pearl Moving Company in Santa Clarita, 91390   [F1=0.459]  (1 iter(s))



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 5: What are the responsibilities of a Senior Planning Engineer in London,...
  Ground truth  : The responsibilities of a Senior Planning Engineer in London, United Kingdom include providing Integ
  Naive RAG     : [F1=0.593]
  Agentic RAG   : Develop and maintain integrated schedules of all Project related activities. Prepare and maintain pr  [F1=0.593]  (1 iter(s))

Avg Word-Overlap F1:
  Naive RAG   : 0.615
  Agentic RAG : 0.615  (Δ +0.000)


## 11a. Understanding RAG with Hybrid Search

### The Complementary Nature of Dense and Sparse Retrieval

| Retrieval type | Strength | Weakness |
|---|---|---|
| **Dense** (Qwen3-Embedding) | Captures semantic meaning; handles paraphrasing | May miss exact keyword matches |
| **Sparse** (BM25) | Precise keyword matching; fast | Ignores synonyms and semantic context |

Neither dominates universally — **hybrid** retrieval captures the best of both.

### Solution: Reciprocal Rank Fusion (RRF)

Rather than normalising and summing raw scores (which requires careful calibration), **RRF** merges ranked lists directly:

$$\text{RRF}(d) = \sum_{r \in \{dense,\,sparse\}} \frac{1}{k + \text{rank}_r(d)}$$

where $k = 10$ is a smoothing constant that prevents top-ranked documents from dominating completely. A document appearing in both lists scores higher than one appearing in only one.

### Pipeline

```
Query
  ├─[Dense: Qwen3-Embedding + ChromaDB] ──▶ Dense top-N   ─┐
  │                                                          ├─[RRF Fusion] ──▶ Merged top-k ──▶ [Gemma] ──▶ Answer
  └─[Sparse: BM25 (rank-bm25)]          ──▶ Sparse top-N  ─┘
```

**`alpha`** controls the blend: `0` = dense-only, `1` = BM25-only, `0.5` = equal weight (default).

**Benefits**: More robust recall; handles both semantic and exact-match queries; no score normalisation  
**Trade-offs**: BM25 index memory; `alpha` tuning; slightly higher retrieval latency

## 11b. Implementing RAG with Hybrid Search

We build a BM25 index over the same chunk texts already stored in ChromaDB, then fuse its rankings with the dense rankings using RRF. `rank-bm25` is already installed from Section 1.

In [ ]:
from rank_bm25 import BM25Okapi

# Build BM25 index over the same chunk_texts used to populate ChromaDB
print("Building BM25 sparse index ...")
_tokenized_corpus = [text.lower().split() for text in chunk_texts]
bm25_index        = BM25Okapi(_tokenized_corpus)
print(f"✓ BM25 index built over {len(chunk_texts)} chunks")


def hybrid_retrieve(query: str, n_results: int = 5, alpha: float = 0.5) -> List[str]:
    """
    Reciprocal Rank Fusion (RRF) of dense and sparse rankings.

    alpha : BM25 weight  (0 = dense-only, 1 = BM25-only, 0.5 = equal)
    k_rrf : RRF smoothing constant (60 per the original paper)
    """
    k_rrf      = 60
    candidates = n_results * 3   # over-fetch from each source before merging

    # Dense retrieval via ChromaDB
    dense_docs = retrieve(query, candidates)["documents"][0]

    # Sparse retrieval via BM25
    bm25_scores  = bm25_index.get_scores(query.lower().split())
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:candidates]
    sparse_docs  = [chunk_texts[i] for i in top_bm25_idx]

    # RRF fusion — no score normalisation needed
    doc_scores: Dict[str, float] = {}
    for rank, doc in enumerate(dense_docs):
        doc_scores[doc] = doc_scores.get(doc, 0.0) + (1 - alpha) / (k_rrf + rank + 1)
    for rank, doc in enumerate(sparse_docs):
        doc_scores[doc] = doc_scores.get(doc, 0.0) + alpha / (k_rrf + rank + 1)

    return sorted(doc_scores, key=lambda d: doc_scores[d], reverse=True)[:n_results]


def rag_with_hybrid_search(query: str, n_results: int = 5, alpha: float = 0.5) -> Dict:
    """
    RAG with Hybrid Search (Dense + BM25 via RRF):
      1. Dense retrieval via Qwen3-Embedding / ChromaDB
      2. Sparse retrieval via BM25
      3. Merge with RRF
      4. Generate answer from fused top-n docs
    """
    retrieved_docs = hybrid_retrieve(query, n_results, alpha)

    context = "\n\n---\n\n".join(retrieved_docs)
    prompt  = f"""Answer the question using only the context provided below.

Context:
{context}

Question: {query}

Answer:"""

    return {
        "query":          query,
        "answer":         generate(prompt),
        "retrieved_docs": retrieved_docs,
        "alpha":          alpha,
    }


# ── Test ────────────────────────────────────────────────────────────────────
result_hybrid = rag_with_hybrid_search(test_query, n_results=5, alpha=0.5)

print(f"Query: {test_query}")
print("=" * 70)
print(f"\n[Hybrid Search RAG] Answer:\n{result_hybrid['answer']}")
print(f"\nBlend: alpha={result_hybrid['alpha']}  (0=dense-only, 1=BM25-only)")
print(f"\nTop fused doc:")
print(f"  {result_hybrid['retrieved_docs'][0][:300]}...")
print()
print("─" * 70)
print("Answer comparison on test query:")
print(f"  Naive RAG     : {result_naive['answer'][:200]}")
print(f"  Hybrid Search : {result_hybrid['answer'][:200]}")

Building BM25 sparse index ...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ BM25 index built over 9295 chunks
Query: Who found the answer to a search query collar george herbert essay?

[Hybrid Search RAG] Answer:
Francisco Rogers found the answer to a search query collar george herbert essay.

Blend: alpha=0.5  (0=dense-only, 1=BM25-only)

Top fused doc:
  Francisco Rogers found the answer to a search query collar george herbert essay
Link ----> collar george herbert essay
Write my essay ESSAYERUDITE.COM
constitution research paper ideas
definition essay humility
business strategy case study solution
corporals course essay
decisions in paradise essays...

──────────────────────────────────────────────────────────────────────
Answer comparison on test query:
  Naive RAG     : Francisco Rogers found the answer to a search query "collar george herbert essay".
  Hybrid Search : Francisco Rogers found the answer to a search query collar george herbert essay.


## 11c. Evaluating Improvement in Results due to Hybrid Search

Final evaluation of Hybrid Search vs Naive RAG, followed by the **complete summary** comparing all six RAG strategies.

In [ ]:
print("Evaluating Naive RAG vs Hybrid Search (5 samples) ...\n")
print("=" * 70)

hybrid_f1s = []

for i, sample in enumerate(EVAL_SAMPLES):
    q        = sample["question"]
    expected = sample["answer"]

    r_hybrid  = rag_with_hybrid_search(q, n_results=5, alpha=0.5)
    f1_hybrid = word_overlap_f1(r_hybrid["answer"], expected)
    f1_naive  = eval_results["Naive RAG"][i]

    hybrid_f1s.append(f1_hybrid)

    print(f"Sample {i+1}: {q[:70]}...")
    print(f"  Ground truth  : {expected[:100]}")
    print(f"  Naive RAG     : [F1={f1_naive:.3f}]")
    print(f"  Hybrid Search : {r_hybrid['answer'][:100]}  [F1={f1_hybrid:.3f}]")
    print()

eval_results["Hybrid Search"] = hybrid_f1s

print("=" * 70)
print(f"Avg Word-Overlap F1:")
print(f"  Naive RAG     : {np.mean(eval_results['Naive RAG']):.3f}")
print(f"  Hybrid Search : {np.mean(hybrid_f1s):.3f}  (Δ {np.mean(hybrid_f1s) - np.mean(eval_results['Naive RAG']):+.3f})")

# ── Complete Final Summary ─────────────────────────────────────────────────
baseline = np.mean(eval_results["Naive RAG"])

print()
print("=" * 70)
print("📊  COMPLETE SUMMARY — Avg Word-Overlap F1 across 5 test samples")
print("=" * 70)
print(f"  {'Method':<35} {'Avg F1':>7}   {'Delta':>8}")
print(f"  {'-'*35}  {'-'*7}   {'-'*8}")
for method, scores in eval_results.items():
    avg   = np.mean(scores)
    delta = f"{avg - baseline:+.3f}" if method != "Naive RAG" else "baseline"
    print(f"  {method:<35} {avg:>7.3f}   {delta:>8}")

print()
print("Note: Word-overlap F1 is a simple lexical proxy metric.")
print("For production evaluation, complement with semantic similarity")
print("(e.g., BERTScore) and human preference judgements.")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating Naive RAG vs Hybrid Search (5 samples) ...



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 1: Who found the answer to a search query collar george herbert essay?...
  Ground truth  : Francisco Rogers found the answer to a search query collar george herbert essay.
  Naive RAG     : [F1=1.000]
  Hybrid Search : Francisco Rogers found the answer to a search query collar george herbert essay.  [F1=1.000]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 2: What are some of the potential negative impacts of charity as discusse...
  Ground truth  : The context discusses that charity can sometimes exacerbate problems rather than solve them. It can 
  Naive RAG     : [F1=0.286]
  Hybrid Search : According to the context, some of the potential negative impacts of charity are:

*   It's a "bandai  [F1=0.354]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 3: Who were the three stars in the NHL game between Buffalo Sabres and Ed...
  Ground truth  : The three stars were Ryan O’Reilly, Brian Gionta, and Leon Draisaitl.
  Naive RAG     : [F1=0.737]
  Hybrid Search : Ryan O’Reilly; Brian Gionta; Leon Draisaitl  [F1=0.737]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sample 4: What services does Pearl Moving Company in Santa Clarita, 91390 offer?...
  Ground truth  : Pearl Moving Company Santa Clarita, 91390 offers services such as apartment moving, local moving, of
  Naive RAG     : [F1=0.459]
  Hybrid Search : The context doesn't explicitly list *all* the services Pearl Moving Company in Santa Clarita, 91390   [F1=0.452]

Sample 5: What are the responsibilities of a Senior Planning Engineer in London,...
  Ground truth  : The responsibilities of a Senior Planning Engineer in London, United Kingdom include providing Integ
  Naive RAG     : [F1=0.593]
  Hybrid Search : The Senior Planning Engineer's responsibilities, according to the provided text, are to:

*   Develo  [F1=0.637]

Avg Word-Overlap F1:
  Naive RAG     : 0.615
  Hybrid Search : 0.636  (Δ +0.021)

📊  COMPLETE SUMMARY — Avg Word-Overlap F1 across 5 test samples
  Method                               Avg F1      Delta
  -----------------------------------  -------   --------
  Naive RAG